# Creating a basic RAG pipeline with Haystack - 02

This notebook is pretty similar to the last one. The main difference is that this time we are serving the embedding model with vLLM.

We will observe the time it takes to generate embeddings a bit further down.

## Setup Environment

### Import packages

In [ ]:
from pathlib import Path
from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import (
    OpenAIDocumentEmbedder, 
    OpenAITextEmbedder
)
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.dataclasses import ChatMessage
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.components.converters import DOCXToDocument
from haystack.utils import Secret
import textwrap
from dotenv import dotenv_values
from transformers import AutoTokenizer
from typing import List
import time
from haystack.utils import Secret

#### Helper Functions

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

def calc_tokens(document:str, sample_text_length:int = 500) -> tuple[int, List[str]]:
    """Calculates the token count of a given text

    Args:
        document (str): The document we want to process.
        sample_text_length (int, optional): How much of that text we want to process. Use -1 to calculate the tokens of the whole text. Defaults to 500.

    Returns:
        tuple[int, List[str]]: First variable is the token count and the second is the actual tokens.
    """

    sample_text = document if sample_text_length == -1 else document[:sample_text_length]

    tokens = tokenizer.tokenize(sample_text)
    token_count = len(tokens)

    return [token_count, tokens]

def convert_documents(documents_dir:Path) -> List[Document]:
    """Return a list of Haystack Documents from a local path that contains our files

    Args:
        documents_dir (Path): The path that contains our files.

    Returns:
        List[Document]: A list that contains the converted Haystack Documents
    """

    FILES = [file.resolve() for file in documents_dir.rglob("*") if file.is_file()]
    converter = DOCXToDocument()

    docs = []
    for file in FILES:
        result = converter.run(sources=[file])
        docs.extend(result["documents"])  # Append the converted documents

    return docs

#### Setup VLLM Connection Parameters

In [ ]:
llm_config = dotenv_values('/nvme/scratch/edu29/llm.env')
embd_config = dotenv_values('/nvme/scratch/edu29/embd.env')

## Indexing Documents and performing RAG

### Setup Indexing components

To be able to index our documents we need to perform 3 things:

1. Split them into smaller chunks so they fit into our embedding model.
2. Turn them into embeddings with an *embedder*.
3. Store them in a Haystack *Document Store* so they can be accessed later on.

For our simple use-case, we will use the bge-base-en-v1.5 document embedder to extract embeddings, and then we will store them in an *InMemoryDocumentStore*.

In [ ]:
DOCUMENTS_DIR = Path("./dummy_data/documents_dir")
docs = convert_documents(documents_dir=DOCUMENTS_DIR)
splitter = DocumentSplitter(split_by="word", split_length=15)
document_store = InMemoryDocumentStore()

In [ ]:
doc_embedder = OpenAIDocumentEmbedder(api_key=Secret.from_token(embd_config['VLLM_API_KEY']),
                                     api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
                                     model=embd_config['MODEL_NAME'],)

#### Why splitting our documents into smaller chunks is important

Lets take a look at our actual text

In [ ]:
print(docs[0].content)

Now lets calculate the tokens of the first chunk (up to the 500th character). Feel free to insert -1 to calculate the tokens for the whole document.

In [ ]:
token_count, tokens = calc_tokens(document=docs[0].content, sample_text_length=500) # Set sample_text_length to -1 to calculate the whole document
print(f'Token Count: {token_count}')
print(f"Tokens: {tokens}")

Now lets try splitting our documents into smaller chunks:

In [ ]:
split_docs = splitter.run(documents=docs)

#### Inspect the document chunk

In [ ]:
chunk_idx = 0 # Change this value to inspect various chunks

for k,v in split_docs['documents'][chunk_idx].meta.items():
    print(f'{k}: {v}')

In [ ]:
token_count, tokens = calc_tokens(document=split_docs['documents'][0].content, sample_text_length=-1)
print(f'Token Count: {token_count}')
print(f'Tokens: {tokens}')

### Extract embeddings and store to Document Store

Go ahead and run the cell below to begin calculating the embeddings.

In [ ]:
split_docs = splitter.run(documents=docs)

start_time = time.time()
docs_with_embeddings = doc_embedder.run(split_docs['documents'])
end_time = time.time()
print(f'Inference time: {end_time - start_time:.2f} seconds')

document_store.write_documents(docs_with_embeddings["documents"])
print(f"Stored {len(docs_with_embeddings['documents'])} documents with embeddings in the document store.")

Compare the running time of the embedder being served by vLLM and Torch.

### Initialize RAG

#### Prompt Template for user

This prompt template will be used by our LLM to generate a response based on our Query.

Specifically, the LLM will read this text from top to bottom:

- It will read the task, which is to respond to the user's query using the **provided context**.
- It will then read some **General Guidelines**.
- Then it will read the **provided context**.
- And finally it will read the **user's query**.

You can see that we pass the **context** and **user's query** through this template. This is why we use a prompt builder later on. This component constructs prompts dynamically by processing chat messages.

Specifically, the *ChatPromptBuilder* component creates prompts using static or dynamic templates written in Jinja2 syntax, by processing a list of chat messages. The templates contain placeholders like {{ variable }} that are filled with values provided during runtime. You can use it for static prompts set at initialization or change the templates and variables dynamically while running.

In [ ]:
with open("/nvme/scratch/edu29/dummy_data/RAG_TEMPLATE.txt") as rag_file:
    rag_template = rag_file.read()

user_prompt_template = ChatMessage.from_user(rag_template)

#### RAG Components

For the actual RAG pipeline we have the following components:

- The *text_embedder* which takes the user's query and turns it into embeddings
- The *retriever* which retrieves the relevant documents
- The *chat_generator* which is our LLM
- The *promt_builder* which was explained above.

In [ ]:
text_embedder = OpenAITextEmbedder(api_key=Secret.from_token(embd_config['VLLM_API_KEY']),
                                     api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
                                     model=embd_config['MODEL_NAME'],)

retriever = InMemoryEmbeddingRetriever(document_store)

chat_generator = OpenAIChatGenerator(api_key=Secret.from_token(llm_config['VLLM_API_KEY']),
                                     api_base_url=f"{llm_config['VLLM_LLM_URL']}/v1",
                                     model=llm_config['MODEL_NAME'],
)
                                                                       
prompt_builder = ChatPromptBuilder(variables=["query", "documents"], required_variables=["query", "documents"])


# Initialize RAG pipeline
basic_rag_pipeline = Pipeline()

basic_rag_pipeline.add_component("text_embedder", text_embedder)
basic_rag_pipeline.add_component("retriever", retriever)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder)
basic_rag_pipeline.add_component("llm", chat_generator)

# Connect the input/output of each component
basic_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
basic_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

We connect each component by defining the inputs and outputs.

For example:

- The *text_embedder* will take the user's query as an input and output *embeddings*. These embeddings will be the input of the *retriever*. The *retriever* will take that input as *query_embedding* and output a list of *documents* that are similar to that query.
- Then, the *retriever* outputs the documents we mentioned, and pass them into our *prompt_builder*.
- Finally, the *prompt_builder* will output the prompt and send it as an input to the *llm*.

#### Perform RAG on our data

Feel free to change the question to something else.

Our documents contain information about the following topics:

- Annual Hackathon the company is organizing
- Cybersecurity Awareness Month
- Employee Recognition Program
- New Office Layout Plan
- Office layout redesign plan
- Product X Launch Timeline
- Product Y Launch Timeline
- QuantumStream product CLI Usage
- QuantumStream product Data Encryption feature
- QuantumStream product Plugin System
- QuantumStream product REST API documentation
- QuantumStream product Scheduler feature
- QuantumStream product Scheduling tasks

---

Feel free to ask anything relating to these topics.

**Suggested prompts:**

- "Whats the purpose of the new office layout? Are we loosing our desks??"
- "I am a new employee at the company. Onboard me about the QuantumStream product."
- "I cannot find the relevant email about the Hackathon, can you tell me more details about it?"

In [ ]:
question = "I am a new employee at the company. Onboard me about the QuantumStream product." # Feel free to change this question

response = basic_rag_pipeline.run(
    data={
        "text_embedder": {"text": question}, 
        "prompt_builder": {"template": [user_prompt_template], "query": question}})
formatted_text = response["llm"]["replies"][0].text
wrapped_text = "\n".join(
    textwrap.fill(line, width=120, subsequent_indent="  ") if line.strip() else line
    for line in formatted_text.splitlines()
)

print(wrapped_text)